# Challenge 03-C-Embedding 

## 1. Overview 

In the last challenge (03-B-Chunking), we worked towards understanding token limits with LLM and utilizing chunking. Now if there are gigabytes of data, we will have lots of chunks to be created as well. Is there a way to select the most relevant chunks of text? The answer is yes. To solve this problem, we can take a look at a process called Embedding. Embedding helps us create numerical representations for all the chunks. Then, we can find the most similar chunks in the the list of embeddings. One popular way to find the similar chunks is through cosine similarity.

### **Embeddings Overview**
An embedding is a special format of data representation that can be easily utilized by machine learning models and algorithms. The embedding is an information dense representation of the semantic meaning of a piece of text. Each embedding is a vector of floating-point numbers, such that the distance between two embeddings in the vector space is correlated with semantic similarity between two inputs in the original format. For example, if two texts are similar, then their vector representations should also be similar.

Different Azure OpenAI embedding models are specifically created to be good at particular tasks:
- Similarity embeddings are good at capturing semantic similarity between two or more pieces of text.
- Text search embeddings help find which long document is relevant to a short query.
- Code search embeddings are useful for embedding code snippets and embedding nature language search queries.

Embeddings make it easier to do machine learning on large inputs representing words by capturing the semantic similarities in a vector space. Therefore, we can use embeddings to if two text chunks are semantically related or similar, and inherently provide a score to assess similarity.

### **Cosine Similarity**
A previously used approach to match similar documents was based on counting maximum number of common words between documents. This is flawed since as the document size increases, the overlap of common words increases even if the topics differ. Therefore cosine similarity is a better approach.

Mathematically, cosine similarity measures the cosine of the angle between two vectors projected in a multi-dimensional space. This is beneficial because if two documents are far apart by Euclidean distance because of size, they could still have a smaller angle between them and therefore higher cosine similarity.

The Azure OpenAI embeddings rely on cosine similarity to compute similarity between documents and a query.

### **Applications**

Embeddings can be created for all different data types including images, audio, video, and text. In this notebook, we will look at generating embeddings for text and csv files. 

There are many applications in which embeddings can be useful. For example, let's say you want to classify a piece of text. Once embeddings are generated, they can be inserted into a machine learning model to predict the right label. In addition, you can utilize embeddings for similarity in time series data, graph data, or for user profile or products. A very popular use case is one that involves semantic search. If you want to retrieve documents that are very relevant to your query, embeddings can be generated for both the query as well as the documents in order to get an accurate response. We will see an example of this in Challenge 4.

## 2. Let's Start Implementation

You will need to import the needed modules. The following cells are key setup steps you completed in the previous challenges.

In [1]:
! pip install num2words
! pip install plotly
! pip install "openai==0.28.1" 
! pip install nptyping


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import openai
import os
import re 
import requests
import sys
from num2words import num2words 
import pandas as pd 
import numpy as np
from openai.embeddings_utils import get_embedding, cosine_similarity 
import tiktoken
from dotenv import load_dotenv
from tenacity import retry, wait_random_exponential, stop_after_attempt
load_dotenv() 

True

Set up your environment to access your Azure OpenAI keys. Refer to your Azure OpenAI resource in the Azure Portal to retrieve information regarding your Azure OpenAI endpoint and keys. 

For security purposes, store your sensitive information in an .env file.

In [3]:
openai.api_type = os.getenv("OPENAI_API_TYPE")
openai.api_key = os.environ.get("OPENAI_API_KEY")
openai.api_base = os.environ.get("OPENAI_API_BASE")
openai.api_version = os.getenv("OPENAI_API_VERSION")
embedding_model=os.getenv("EMBEDDING_MODEL_NAME")

## 3. Generate Embeddings on text

#### Student Task #1:
Use the Azure OpenAI Embeddings class to create an embedding for the input text below. 

In [4]:

embedding = openai.Embedding.create(
    input="I would like to order a pizza", engine=embedding_model
)["data"][0]["embedding"]
print(embedding)
len(embedding)

[0.006326164584606886, -0.013710793107748032, -0.013661562465131283, -0.01329233031719923, -0.02008618786931038, 0.008215398527681828, -0.012098482809960842, -0.005215393844991922, 0.007267704699188471, -0.018006185069680214, 0.03421544283628464, -0.005756933242082596, 0.0047415466979146, -0.008424630388617516, 0.009815401397645473, -0.006375395692884922, 0.02995697408914566, -0.00972309336066246, 0.011329250410199165, -0.0076615517027676105, 0.00014836563786957413, -0.013747716322541237, 0.01575387269258499, -0.008633861318230629, -0.016861567273736, 0.0029430820140987635, 0.004076930228620768, 0.010929249227046967, -0.004535392392426729, -0.004550776910036802, 0.04219084233045578, 0.015224642120301723, -0.01734156906604767, -0.025193890556693077, -0.017760030925273895, 0.002500004367902875, -0.0026800045743584633, -0.010215402580797672, -0.002460004296153784, -0.028750818222761154, 0.018313877284526825, 0.003944622352719307, 0.004932316020131111, -0.036972369998693466, -0.02759389393

1536

The openai.Embedding.create() method will take a list of text - here we have a single sentence - and then will return a list containing a single embedding. You can use these embeddings when searching, providing recommendations, classification, and more.

### 3.1 Generate Embeddings for a CSV file

#### Student Task #2:
Enter in the path of the `Automobile.csv` file which you can find in the `/data` folder. Run the cells below.

In [ ]:
df=pd.read_csv(os.path.join(os.getcwd(),r'Enter path here'))
df

In [ ]:
shortened_df = df[['name', 'mpg', 'origin']]
shortened_df

In [ ]:
tokenizer = tiktoken.get_encoding("cl100k_base")
shortened_df['n_tokens'] = shortened_df["name"].apply(lambda x: len(tokenizer.encode(x)))
shortened_df = shortened_df[shortened_df.n_tokens<8192]
len(shortened_df)

In [ ]:
shortened_df

In [ ]:
sample_encode = tokenizer.encode(shortened_df.name[0]) 
decode = tokenizer.decode_tokens_bytes(sample_encode)
decode

In [ ]:
len(decode)
shortened_df['ada-v2'] = shortened_df['name'].apply(lambda x : get_embedding(x, engine = embedding_model)) 

In [ ]:
shortened_df

The embeddings generated from the csv file can be used to perform search. You can calculate the cosine similarity between a query embedding and the embeddings from the csv file. Then you can rank the search results to what is most relevant to the query. We will see an application of embedddings in Challenge 4.

## Success Criteria 

To complete this challenge successfully:

* Show an understanding of embeddings by working with different inputs.